# Pipeline de limpieza de ESCO

El primer paso es importar las librerias que se van a utilizar.

In [ ]:
from pathlib import Path

import pandas as pd
# Se elige como se ven las tablas
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

## Carga de datos

In [3]:
ruta_datos = Path("../data/raw/esco") # Definir la ruta de los archivos

occupations_original = pd.read_csv(ruta_datos / "occupations_es.csv", encoding="utf-8")
skills_original = pd.read_csv(ruta_datos / "skills_es.csv", encoding="utf-8")
occupation_skill_relations_original = pd.read_csv(ruta_datos / "occupationSkillRelations_es.csv", encoding="utf-8")

# Copias de trabajo para aplicar las transformaciones
occupations = occupations_original.copy()
skills = skills_original.copy()
occupation_skill_relations = occupation_skill_relations_original.copy()

## Transformaciones

### Fechas

La columna `modifiedDate` de los conjuntos de datos `occupations` y `skills` contiene fechas almacenadas como cadenas de texto. Se convierten al tipo datetime para facilitar su tratamiento y permitir operaciones cronológicas.

Se utiliza el formato ISO 8601 y se establece UTC como zona horaria.

In [4]:
for nombre, df in {"occupations": occupations, "skills": skills}.items():
    df["modifiedDate"] = pd.to_datetime(df["modifiedDate"], format="ISO8601", errors="raise", utc=True)

### Espacios sobrantes

Se detectaron espacios en blanco sobrantes al principio o al final de algunos valores de texto. Se corrige ahora esta inconsistencia.

In [5]:
# Columnas afectadas de occupations
occupations["description"] = (occupations["description"].str.strip())

# Columnas afectadas de skills
skills["preferredLabel"] = (skills["preferredLabel"].str.strip())

skills["description"] = (skills["description"].str.strip())

### Valores duplicados

Durante el análisis exploratorio se detectaron URI repetidas en los conjuntos de datos `occupations` y `skills`. En ambos casos, los registros asociados a una misma URI presentaban diferencias únicamente en la columna `modifiedDate`.

Se decide conservar el registro con la fecha de modificación más reciente para cada URI y eliminar las versiones anteriores, evitando mantener distintas versiones del mismo concepto.

In [6]:
# Ordenar por fecha de modificación, de más reciente a más antigua
occupations = occupations.sort_values(by="modifiedDate", ascending=False)
skills = skills.sort_values(by="modifiedDate", ascending=False)

# Conservar únicamente el registro más reciente de cada URI
occupations = occupations.drop_duplicates(subset="conceptUri", keep="first").reset_index(drop=True)
skills = skills.drop_duplicates(subset="conceptUri", keep="first").reset_index(drop=True)

De otra forma, se vió que en `skills` habían 3 registros con el campo `preferredLabel` duplicado que no tenían relaciones. Se decide eliminar esos registros puesto que no aportan información.

In [ ]:
# Identificar las URI de skills que aparecen en las relaciones
skills_con_relaciones = occupation_skill_relations["skillUri"].unique()

# Denominaciones que queremos revisar
labels_duplicadas = ["asumir la responsabilidad", "calcular probabilidades", "improvisar"]

# Filtrar los registros de esas denominaciones que no tienen relaciones
mascara = (skills["preferredLabel"].isin(labels_duplicadas) & ~skills["conceptUri"].isin(skills_con_relaciones))

# Eliminar los registros
skills = skills[~mascara].reset_index(drop=True)

### Valores ausentes

Se identificaron las columnas `scopeNote` y `definition` con un 100 % de valores ausentes en los conjuntos de datos analizados.

Al no contener información en ninguno de sus registros, se decide eliminar ambas columnas para evitar conservar variables que no aportan información útil.

In [8]:
columnas_eliminar = ["scopeNote", "definition"]

occupations = occupations.drop(columns=columnas_eliminar)
skills = skills.drop(columns=columnas_eliminar)

Después se tratan los valores ausentes encontrados en `skillType` y `reuseLevel` en `skills`. Como esos cinco registros formaban parten de las competencias digiComp, y los otros registros de las mismas competencias con dichos valores no nulos tenían todos el mismo valor, se decide rellenar estos con ese mismo valor.

In [9]:
# Identificar registros con valores ausentes
mascara = skills["skillType"].isna() | skills["reuseLevel"].isna()

# Rellenar los valores ausentes
skills.loc[mascara, "skillType"] = (skills.loc[mascara, "skillType"].fillna("skill/competence"))

skills.loc[mascara, "reuseLevel"] = (skills.loc[mascara, "reuseLevel"].fillna("cross-sector"))

In [10]:
# Crear un diccionario que relaciona cada skill_uri con su skill_type
diccionario_skill_type = skills.set_index("conceptUri")["skillType"]

# Completar los valores ausentes en occupation_skill_relations
mascara = occupation_skill_relations["skillType"].isna()

occupation_skill_relations.loc[mascara, "skillType"] = (
    occupation_skill_relations.loc[mascara, "skillUri"]
    .map(diccionario_skill_type)
)

Por otras parte, se decide combinar las columnas `altLabels` y `hiddenLabels` en una única columna `altLabels`, ya que ambas contienen denominaciones alternativas. De esta manera, se centralizan en un solo campo y se elimina la columna `hiddenLabels`, evitando mantener información separada con la misma finalidad.

In [11]:
for nombre in ["occupations", "skills"]:
    df = globals()[nombre]

    # Combinar las denominaciones alternativas, respetando los saltos de línea
    df["altLabels"] = (
        df["altLabels"].fillna("").str.strip()
        + "\n"
        + df["hiddenLabels"].fillna("").str.strip()
    ).str.strip().replace("", pd.NA)

    # Eliminar la columna hiddenLabels
    df = df.drop(columns="hiddenLabels")

    globals()[nombre] = df

### Columnas sin información

Se vio también que habían columnas en `occupations` y en `skills` que tan solo repetían un mismo valor en todos los registros, `conceptType` y `status`. Como no aportan información, se decide deshacerse de las mismas.

Por otra parte, tras hacer la limpieza, la columna `modifiedDate` ya no aporta información necesaria.

In [12]:
occupations = occupations.drop(columns=["conceptType", "status", "modifiedDate"])
skills = skills.drop(columns=["conceptType", "status", "modifiedDate"])

### Formato nombre columnas

Se cambia el nombre de las columnas para que estén en snake_case.

In [ ]:
for df in [occupations, skills, occupation_skill_relations]:
    df.columns = (df.columns.str.replace(r"([a-z0-9])([A-Z])", r"\1_\2", regex=True).str.lower())

### Categóricas

Como se ha observado que hay variables categóricas, aunque se estén considerando como texto, se pasan a categóricas para ahorrar memoria.

In [ ]:
# Columnas categóricas de occupations
occupations_categoricas = ["regulated_profession_note", "in_scheme"]

# Columnas categóricas de skills
skills_categoricas = ["skill_type", "reuse_level", "in_scheme"]

# Columnas categóricas de occupation_skill_relations
relations_categoricas = ["relation_type", "skill_type"]

# Convertir a category
occupations[occupations_categoricas] = (occupations[occupations_categoricas].astype("category"))
skills[skills_categoricas] = (skills[skills_categoricas].astype("category"))
occupation_skill_relations[relations_categoricas] = (occupation_skill_relations[relations_categoricas].astype("category"))

## Validación

### Nombres columnas

Primero se comprueba que se ha cambiado correctamente los nombres de las columnas, porque es lo que se va a utilizar para llamarlas.

In [15]:
for nombre, df in {
    "occupations": occupations,
    "skills": skills,
    "occupation_skill_relations": occupation_skill_relations
}.items():
    print(f"\n{nombre}:")
    print(df.columns.tolist())


occupations:
['concept_uri', 'isco_group', 'preferred_label', 'alt_labels', 'regulated_profession_note', 'in_scheme', 'description', 'code', 'nace_code']

skills:
['concept_uri', 'skill_type', 'reuse_level', 'preferred_label', 'alt_labels', 'in_scheme', 'description']

occupation_skill_relations:
['occupation_uri', 'occupation_label', 'relation_type', 'skill_type', 'skill_uri', 'skill_label']


### Espacios sobrantes

Se comprueba que ya no quedan espacios sobrantes.

In [16]:
columnas_a_comprobar = {
    "occupations": {
        "df": occupations,
        "columnas": ["description"]
    },
    "skills": {
        "df": skills,
        "columnas": ["preferred_label", "description"]
    }
}

for nombre, datos in columnas_a_comprobar.items():
    print(f"\n{nombre}")
    print("-" * 50)

    for columna in datos["columnas"]:
        valores = datos["df"][columna].dropna().astype(str)
        espacios = valores.str.strip().ne(valores).sum()

        print(f"{columna}: {espacios} registros con espacios sobrantes")


occupations
--------------------------------------------------
description: 0 registros con espacios sobrantes

skills
--------------------------------------------------
preferred_label: 0 registros con espacios sobrantes
description: 0 registros con espacios sobrantes


### Valores ausentes

Se comprueba que, al eliminar columnas, han cambiado las dimensiones de las muestras.

In [17]:
for nombre, df in {"occupations": occupations, "skills": skills}.items():
    print(f"\n{nombre}")
    print("-" * 40)
    print("Dimensiones:", df.shape)
    # columnas occupations: 15 - 2 vacias - hiddenLabels - 2 de un solo valor - modifiedDate
    # columnas skills: 13 - 2 vacias - hiddenLabels - 2 de un solo valor - modifiedDate


occupations
----------------------------------------
Dimensiones: (3039, 9)

skills
----------------------------------------
Dimensiones: (13936, 7)


Luego se comprueba que ya no queda valores ausentes en las columnas `skillType` y `reuseLevel` de `skills`.

In [18]:
print("Valores ausentes en skill_type de skills:", skills["skill_type"].isna().sum())
print("Valores ausentes en reuse_level de skills:", skills["reuse_level"].isna().sum())
print(f"Valores ausentes en skill_type de occupation_skill_relations: {occupation_skill_relations["skill_type"].isna().sum()}")

Valores ausentes en skill_type de skills: 0
Valores ausentes en reuse_level de skills: 0
Valores ausentes en skill_type de occupation_skill_relations: 0


### Valores duplicados

Se comprueba que ya no quedan Uris repetidos.

In [19]:
for nombre, df, df_original in [
    ("occupations", occupations, occupations_original),
    ("skills", skills, skills_original)
]:

    registros_eliminados = len(df_original) - len(df)
    uri_duplicadas = df["concept_uri"].duplicated().sum()

    print(f"\n{nombre}")
    print("-" * 45)
    print(f"Registros iniciales: {len(df_original):,}")
    print(f"Registros finales: {len(df):,}")
    print(f"Registros eliminados: {registros_eliminados}")
    print(f"URI duplicadas restantes: {uri_duplicadas}")


occupations
---------------------------------------------
Registros iniciales: 3,043
Registros finales: 3,039
Registros eliminados: 4
URI duplicadas restantes: 0

skills
---------------------------------------------
Registros iniciales: 13,960
Registros finales: 13,936
Registros eliminados: 24
URI duplicadas restantes: 0


Se comprueban luego los duplicados que han quedado en `preferredLabel` de `skills` tras eliminar algunos registros.

In [20]:
print("Duplicados por conceptUri:", skills["concept_uri"].duplicated().sum())
print("Duplicados por preferredLabel:", skills["preferred_label"].duplicated().sum()) #41 iniciales - 21 Uris repetidos - 3 sin relaciones = 17 
print("Registros completamente duplicados:", skills.duplicated().sum())

Duplicados por conceptUri: 0
Duplicados por preferredLabel: 17
Registros completamente duplicados: 0


Despues de haber eliminado registros, se comprueba que no se ha eliminado ninguno que no correspondía. Es decir, se comprueba que todos los Uris en occupationSkillRelations sigue correspondiendose con un Uri de occupations o de skills.

In [21]:
# Comprobar URI de ocupaciones
uris_occupations = set(occupations["concept_uri"])
uris_relaciones_occupations = set(occupation_skill_relations["occupation_uri"])

ocupaciones_sin_correspondencia = uris_relaciones_occupations - uris_occupations


# Comprobar URI de competencias
uris_skills = set(skills["concept_uri"])
uris_relaciones_skills = set(occupation_skill_relations["skill_uri"])

skills_sin_correspondencia = uris_relaciones_skills - uris_skills


# Mostrar resultados
print("URI de ocupaciones en relaciones sin correspondencia:", len(ocupaciones_sin_correspondencia))
print("URI de competencias en relaciones sin correspondencia:", len(skills_sin_correspondencia))

URI de ocupaciones en relaciones sin correspondencia: 0
URI de competencias en relaciones sin correspondencia: 0


### Tipos de datos

Finalmente se comprueba que no haya habido ningún error, cambiando tipos de datos sin querer.

In [25]:
# Revisar tipos de datos de los tres DataFrames
for nombre, df in {
    "occupations": occupations,
    "skills": skills,
    "occupation_skill_relations": occupation_skill_relations
}.items():

    print(f"\n{nombre}")
    print("-" * 60)
    df.info()


occupations
------------------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 3039 entries, 0 to 3038
Data columns (total 9 columns):
 #   Column                     Non-Null Count  Dtype   
---  ------                     --------------  -----   
 0   concept_uri                3039 non-null   str     
 1   isco_group                 3039 non-null   int64   
 2   preferred_label            3039 non-null   str     
 3   alt_labels                 2899 non-null   str     
 4   regulated_profession_note  3039 non-null   category
 5   in_scheme                  3039 non-null   category
 6   description                3039 non-null   str     
 7   code                       3039 non-null   str     
 8   nace_code                  3039 non-null   str     
dtypes: category(2), int64(1), str(6)
memory usage: 172.3 KB

skills
------------------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 13936 entries, 0 to 13935
Data colu

## Salida

Finalmente, se guardan los cambios de la capa silver.

In [23]:
ruta_silver = Path("../data/silver/esco") # Definir la ruta de destino

# Crear la carpeta si todavía no existe
ruta_silver.mkdir(parents=True, exist_ok=True)

# Guardar los DataFrames limpios
occupations.to_csv(ruta_silver / "occupations_es_silver.csv", index=False, encoding="utf-8")
skills.to_csv(ruta_silver / "skills_es_silver.csv", index=False, encoding="utf-8")
occupation_skill_relations.to_csv(ruta_silver / "occupation_skill_relations_es_silver.csv", index=False, encoding="utf-8")

print("Los tres archivos se han guardado correctamente.")

Los tres archivos se han guardado correctamente.


## Resultados y conclusiones

### Comparación de los datos antes y después de la limpieza

Se ha realizado un proceso de análisis y limpieza de los tres conjuntos de datos de ESCO: ocupaciones (`occupations`), habilidades (`skills`) y relaciones entre ocupaciones y habilidades (`occupationSkillRelations`).

In [30]:
datasets = {
    "occupations": (occupations_original, occupations),
    "skills": (skills_original, skills),
    "occupation_skill_relations": (
        occupation_skill_relations_original,
        occupation_skill_relations
    )
}

comparacion = pd.DataFrame({
    nombre: {
        "registros_iniciales": len(original),
        "registros_finales": len(limpio),
        "diferencia_registros": len(limpio) - len(original),
        "columnas_iniciales": original.shape[1],
        "columnas_finales": limpio.shape[1],
        "diferencia_columnas": limpio.shape[1] - original.shape[1]
    }
    for nombre, (original, limpio) in datasets.items()
}).T

display(comparacion)

,registros_iniciales,registros_finales,diferencia_registros,columnas_iniciales,columnas_finales,diferencia_columnas
occupations,3043,3039,-4,15,9,-6
skills,13960,13936,-24,13,7,-6
occupation_skill_relations,126051,126051,0,6,6,0


### Problemas encontrados y acciones realizadas

| Problema detectado | Acción realizada |
|---|---|
| Columnas completamente vacías (`scopeNote` y `definition`) | Eliminación de las columnas. |
| Valores ausentes en `skillType` y `reuseLevel` de cinco competencias | Imputación utilizando los valores mayoritarios de conceptos DigComp comparables. |
| URI duplicadas en `occupations` y `skills` | Conservación del registro con la fecha de modificación más reciente. |
| Tres competencias con denominaciones repetidas y URI distintas, una de las versiones sin relaciones | Eliminación de las tres versiones sin relaciones. |
| Espacios en blanco al principio y al final de algunos campos | Eliminación de espacios sobrantes. |
| Información repartida entre `altLabels` y `hiddenLabels` | Combinación en una única columna `altLabels`. |
| Columnas constantes (`conceptType` y `status`) | Eliminación por no aportar variabilidad. |
| Tipos de datos poco adecuados | Conversión de las columnas categóricas seleccionadas al tipo `category`. |

También se comprobó que todas las URI utilizadas en las relaciones tienen correspondencia en los conjuntos de ocupaciones y competencias, garantizando la integridad referencial entre los tres datasets.


### Utilidad de los datos para el proyecto

La clasificación ESCO proporciona una base estructurada para relacionar ocupaciones profesionales con las competencias necesarias para desempeñarlas.

- `occupations`: contiene las ocupaciones.
- `skills`: recoge las habilidades y conocimientos.
- `occupationSkillRelations`: establece las relaciones entre ocupaciones y competencias, diferenciando entre competencias esenciales y opcionales.
Esta estructura permite construir un modelo de datos que relacione las ocupaciones con las competencias requeridas y que pueda servir como base para funcionalidades de identificación de perfiles profesionales, análisis de competencias y orientación laboral.

Por tanto, los datos resultan adecuados para representar las relaciones entre ocupaciones y habilidades, que constituyen una parte fundamental de la idea del proyecto.

### Información que falta y limitaciones

A pesar de su utilidad, ESCO presenta algunas limitaciones importantes.

ESCO es una clasificación de ocupaciones y competencias, no una base de datos de vacantes laborales ni de candidatos. Por tanto, no permite conocer directamente:
- Qué ocupaciones tienen más demanda actualmente.
- Qué competencias solicitan las empresas en sus ofertas.
- Los salarios asociados a cada ocupación.
- La localización geográfica de las oportunidades laborales.
- La experiencia o formación concreta exigida por las empresas.

Limitaciones de cobertura y posibles sesgos:
- La clasificación representa el mercado laboral europeo y no necesariamente las particularidades de todos los mercados laborales nacionales o locales.
- Las competencias asociadas a cada ocupación son una representación normalizada y no tienen por qué coincidir con los requisitos de todas las empresas.
- La clasificación puede no reflejar inmediatamente la aparición de nuevas profesiones y competencias o los cambios en las existentes.
- La asignación de competencias esenciales y opcionales responde a la estructura de ESCO y no necesariamente a las necesidades particulares de cada puesto de trabajo.